# 08 - Model Comparison & Analysis

Comparative analysis of all 5 models: DNN, CNN, LSTM, Transformer, and Multi-Modal.
This notebook addresses the **Experimental & Comparative Analysis** criterion (2 marks).

In [ ]:
import numpy as np
import tensorflow as tf
import keras
import pandas as pd
import matplotlib.pyplot as plt

# Define custom layers for model loading
@keras.saving.register_keras_serializable()
class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, max_len, embed_dim, **kwargs):
        super(PositionalEncoding, self).__init__(**kwargs)
        self.max_len = max_len
        self.embed_dim = embed_dim
        pos = np.arange(max_len)[:, np.newaxis]
        i = np.arange(embed_dim)[np.newaxis, :]
        angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(embed_dim))
        angle_rads = pos * angle_rates
        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
        self.pos_encoding = tf.constant(angle_rads, dtype=tf.float32)

    def call(self, inputs):
        return inputs + self.pos_encoding[:tf.shape(inputs)[1], :]

    def get_config(self):
        config = super().get_config()
        config.update({'max_len': self.max_len, 'embed_dim': self.embed_dim})
        return config

@keras.saving.register_keras_serializable()
class TransformerEncoder(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super(TransformerEncoder, self).__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.rate = rate
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation="relu"),
            tf.keras.layers.Dense(embed_dim),
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

    def get_config(self):
        config = super().get_config()
        config.update({'embed_dim': self.embed_dim, 'num_heads': self.num_heads, 'ff_dim': self.ff_dim, 'rate': self.rate})
        return config

print('Libraries imported')

## Load All Trained Models

In [ ]:
# Load models
dnn_model = tf.keras.models.load_model('dataset/processed/dnn_model.keras')
cnn_model = tf.keras.models.load_model('dataset/processed/cnn_model.keras')
lstm_model = tf.keras.models.load_model('dataset/processed/lstm_model.keras')
try:
    transformer_model = tf.keras.models.load_model('dataset/processed/transformer_model.keras')
except:
    transformer_model = tf.keras.models.load_model(
        'dataset/processed/transformer_model.keras',
        custom_objects={'PositionalEncoding': PositionalEncoding}
    )
multimodal_model = tf.keras.models.load_model('dataset/processed/multimodal_model.keras')

print('All 5 models loaded')

## Load Test Data

In [ ]:
# Load tabular test data (DNN, Multi-Modal)
X_tab_test = np.load('dataset/processed/X_tab_test.npy')
y_tab_test = np.load('dataset/processed/y_tab_test.npy')

# Load image test data (CNN, Multi-Modal)
X_img_test = np.load('dataset/processed/X_img_test.npy')
y_img_test = np.load('dataset/processed/y_img_test.npy')

# Load text test data (LSTM, Transformer)
X_text_test = np.load('dataset/processed/X_text_test.npy')
y_text_test = np.load('dataset/processed/y_text_test.npy')

print('Test data loaded')

## Model Comparison

In [ ]:
# Evaluate all models
results = []

# 1. DNN - Tabular -> Rating Regression
dnn_loss, dnn_mae = dnn_model.evaluate(X_tab_test, y_tab_test, verbose=0)
dnn_params = dnn_model.count_params()
results.append({
    'Model': 'DNN',
    'Type': 'Regression (Rating)',
    'Input': 'Tabular (4 features)',
    'Params': f'{dnn_params:,}',
    'MSE': f'{dnn_loss:.4f}',
    'MAE': f'{dnn_mae:.4f}',
    'Accuracy': 'N/A'
})

# 2. CNN - Image -> High/Low Tier Classification
cnn_loss, cnn_acc = cnn_model.evaluate(X_img_test, y_img_test, verbose=0)
cnn_params = cnn_model.count_params()
results.append({
    'Model': 'CNN (MobileNetV2)',
    'Type': 'Binary Classification',
    'Input': 'Images (224x224x3)',
    'Params': f'{cnn_params:,}',
    'MSE': 'N/A',
    'MAE': 'N/A',
    'Accuracy': f'{cnn_acc:.2%}'
})

# 3. LSTM - Text -> Sentiment Classification
lstm_loss, lstm_acc = lstm_model.evaluate(X_text_test, y_text_test, verbose=0)
lstm_params = lstm_model.count_params()
results.append({
    'Model': 'LSTM (Bidirectional)',
    'Type': '3-Class Sentiment',
    'Input': 'Text (tokenized, 100 tokens)',
    'Params': f'{lstm_params:,}',
    'MSE': 'N/A',
    'MAE': 'N/A',
    'Accuracy': f'{lstm_acc:.2%}'
})

# 4. Transformer - Text -> Sentiment Classification
trans_loss, trans_acc = transformer_model.evaluate(X_text_test, y_text_test, verbose=0)
trans_params = transformer_model.count_params()
results.append({
    'Model': 'Transformer',
    'Type': '3-Class Sentiment',
    'Input': 'Text (tokenized + positional encoding)',
    'Params': f'{trans_params:,}',
    'MSE': 'N/A',
    'MAE': 'N/A',
    'Accuracy': f'{trans_acc:.2%}'
})

# 5. Multi-Modal - Images + Tabular -> Rating Regression
mm_loss, mm_mae = multimodal_model.evaluate([X_img_test, X_tab_test], y_tab_test, verbose=0)
mm_params = multimodal_model.count_params()
results.append({
    'Model': 'Multi-Modal',
    'Type': 'Regression (Rating)',
    'Input': 'Images + Tabular',
    'Params': f'{mm_params:,}',
    'MSE': f'{mm_loss:.4f}',
    'MAE': f'{mm_mae:.4f}',
    'Accuracy': 'N/A'
})

# Display comparison table
comparison_df = pd.DataFrame(results)
print('=' * 100)
print('MODEL COMPARISON TABLE')
print('=' * 100)
print(comparison_df.to_string(index=False))
print('=' * 100)

## Visual Comparison

In [ ]:
# Bar chart comparing model metrics
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

model_names = ['DNN', 'CNN', 'LSTM', 'Transformer', 'Multi-Modal']

# Accuracy comparison (classification models only)
accuracies = [None, cnn_acc, lstm_acc, trans_acc, None]
valid_acc = [(m, a) for m, a in zip(model_names, accuracies) if a is not None]
if valid_acc:
    axes[0].bar([m for m, a in valid_acc], [a for m, a in valid_acc], color=['blue', 'green', 'red'])
    axes[0].set_title('Classification Accuracy')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_ylim(0, 1)
    for i, (m, a) in enumerate(valid_acc):
        axes[0].text(i, a + 0.01, f'{a:.2%}', ha='center')

# MAE comparison (regression models only)
maes = [dnn_mae, None, None, None, mm_mae]
valid_mae = [(m, v) for m, v in zip(model_names, maes) if v is not None]
if valid_mae:
    axes[1].bar([m for m, v in valid_mae], [v for m, v in valid_mae], color=['orange', 'purple'])
    axes[1].set_title('Regression MAE (lower is better)')
    axes[1].set_ylabel('MAE')
    for i, (m, v) in enumerate(valid_mae):
        axes[1].text(i, v + 0.01, f'{v:.4f}', ha='center')

# Parameter count comparison
param_counts = [dnn_params, cnn_params, lstm_params, trans_params, mm_params]
axes[2].bar(model_names, param_counts, color='teal')
axes[2].set_title('Model Size (Parameters)')
axes[2].set_ylabel('Parameter Count')
for i, v in enumerate(param_counts):
    axes[2].text(i, v + max(param_counts)*0.01, f'{v:,}', ha='center', rotation=45)

plt.tight_layout()
plt.show()

## Comparative Analysis

### Strengths & Weaknesses

| Model | Strengths | Weaknesses |
|-------|-----------|------------|
| **DNN** | Simple, fast, interpretable features | Limited to tabular data, no spatial/text understanding |
| **CNN** | Transfer learning, visual pattern recognition | Needs large image dataset, images uncorrelated with rating |
| **LSTM** | Handles sequential text, bidirectional context | Slower than Transformer, vanishing gradient on long sequences |
| **Transformer** | Parallel processing, long-range dependencies | Needs more data than LSTM, missing positional encoding hurts |
| **Multi-Modal** | Combines multiple data types, real-world assessment | Complex training, two-branch co-training challenges |

### Key Insights

1. **Data quality > Model complexity**: The CNN (most complex model) performs worst because images don't correlate with restaurant quality
2. **Text models perform well**: Both LSTM and Transformer achieve high accuracy on sentiment, but the small review set (15 templates) inflates results
3. **Multi-modal fusion works**: Multi-Modal MAE is close to DNN MAE, showing that adding image features doesn't hurt but doesn't help much with current data
4. **Simplicity wins**: The DNN with only 4 features achieves reasonable MAE despite no access to images or text

### Recommendations

1. **Expand dataset**: 150 samples is too small for meaningful deep learning. Target 500+ restaurants
2. **Diverse reviews**: Generate unique reviews per restaurant using templates with random variation
3. **Better image-task correlation**: Use restaurant-specific images (interior, exterior) not generic food photos
4. **Hyperparameter tuning**: Each model should be tested with at least 3 different configurations
5. **Cross-validation**: Use 5-fold CV to get more reliable performance estimates

## Conclusion

This project demonstrates 5 different deep learning architectures on a multi-modal restaurant dataset.
The DNN and Multi-Modal models tackle rating regression, while CNN handles image classification,
and LSTM/Transformer handle sentiment analysis. The multi-modal approach combining images and
tabular data is the most innovative aspect, mimicking how humans assess restaurant quality by
considering both visual appearance and structured information.

**Note**: To improve results, the dataset should be expanded with more diverse restaurant data,
actual web-scraped content, and more unique review texts per restaurant.